# CarRacing · 연속 Beta-PPO
GPU 런타임에서 순서대로 실행하세요. 수업에서 받은 한국어/영어 합본 carracing_competition.zip을 준비하세요. 학습 기본값은 seed 42, 원본 300,000프레임입니다. Drive의 전용 경로에 기록합니다. 동일 run의 재시작은 공식 latest.pt부터 이어지며 세션 종료 전 마지막 저장 이후 구간은 재실행됩니다. 최종 재현은 깨끗한 폴더에서 한 번에 실행하세요.

[CarRacing Beta-PPO 논문](https://arxiv.org/abs/2111.02202)의 bounded policy 아이디어를 참고해 새로 구현했습니다. 정책은 조향과 부호 있는 페달 강도를 연속적으로 선택합니다. 양수는 가속, 음수는 제동입니다. 학습에는 샘플을, 평가에는 동일 분포의 평균을 사용합니다.

In [ ]:
import os, sys, subprocess, hashlib, zipfile, shutil
from pathlib import Path
from google.colab import drive, files
drive.mount('/content/drive')
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
root = Path('/content/beta_ppo_official')
run = Path('/content/drive/MyDrive/CarRacing/beta_ppo_s42_v1')
zip_path = Path('/content/carracing_competition.zip')
if not zip_path.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('공식 ZIP 하나를 선택하세요.')
    zip_path.write_bytes(next(iter(uploaded.values())))
manifest = {'agents/actor_critic.py': 'd25aa847bf5d9c0c9803c0016849f2d9bae410caeecf0d943fba8fd695c02037', 'agents/base.py': 'f183f1e46745a9abb5e7f0fc88a08be134104f599326f57ac9bc7f63a324e743', 'agents/dqn.py': '37d7168f7bcb252eff69aab561e85c5ab3d73f01007f6ab95ab00d60fd0d5415', 'agents/__init__.py': '215cd1e434eaf15791c2c826380c5eab6844225ce7b5b9e134894eefd52edc59', 'buffers.py': '59b576518a9997a51f2811f00c8c900394dba24f45d6fb8eaa05f0324da018f1', 'check_submission.py': '7a62312300656f891d1aeffdcab249c501de0153567138f027e8491998763a02', 'envs.py': 'ceee2d847af1a3a498c6b7f0f83f3561b24ac22e53de8bded43c97a15af645d0', 'evaluate.py': '219d4c3ded5b8624d8e4f69d3c71e27145aed6b9eb20dc0ef8b0f9c2355c2574', 'export.py': 'dbd9fe51c3046d2d5a458ad8ea464e99cde777c13be7d8406bf2f667b05f8a6f', 'networks.py': '05c5c6dbd6f4ce2fbfe2a71ef18f84d175ff7a7ff36540c9172f750f27b2b3d8', 'README.md': 'f4b928b8d1ae7f907d21670f5d5c0fe6050845f056ab567ab7b18968608a86b0', 'requirements.txt': 'b38b63f4f8d53b8edce022020a26886d6f079d3484d16d3eb630e530fd11b9b7', 'train.py': '579b46fbf559233490bd3e0949b29880dfe564d890e15c37ba99db7a2344b6b1'}
with zipfile.ZipFile(zip_path) as archive:
    for name, digest in manifest.items():
        data = archive.read('carracing_competition/' + name)
        if hashlib.sha256(data).hexdigest() != digest:
            raise ValueError(f'공식 배포본 버전을 확인하세요: {name}')
        dest = root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
os.chdir(root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

In [ ]:
%%writefile agents/beta_ppo.py
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.distributions import Beta
from agents.base import Agent
from networks import CNNEncoder


def controls(sample):
    steer, pedal = (2 * sample - 1).unbind(-1)
    return torch.stack((steer, pedal.relu(), (-pedal).relu()), dim=-1)


class BetaPolicy(nn.Module):
    def __init__(self, shape):
        super().__init__()
        self.encoder = CNNEncoder(shape, feature_dim=256)
        self.actor = nn.Linear(256, 4)
        self.critic = nn.Linear(256, 1)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        # Mild forward exploration at initialization. All four biases are trainable.
        with torch.no_grad():
            self.actor.bias.copy_(torch.tensor([0., 1.5, 0., -0.5]))

    def parameters_at(self, obs):
        features = self.encoder(obs)
        alpha, beta = (1 + F.softplus(self.actor(features))).chunk(2, dim=-1)
        return alpha, beta, self.critic(features).squeeze(-1)

    def forward(self, obs):
        alpha, beta, _ = self.parameters_at(obs)
        return controls(alpha / (alpha + beta))


class BetaPPO(Agent):
    name = "beta_ppo"
    default_config = dict(continuous=True, lr=0.00025, gamma=0.99, gae_lambda=0.95,
                          rollout=512, epochs=4, batch=64, clip=0.2, reward_scale=0.1,
                          entropy_coef=0.005, target_kl=0.025)

    def __init__(self, obs_shape, action_space, config, device):
        super().__init__(obs_shape, action_space, config, device)
        if not self.continuous:
            raise ValueError("beta_ppo requires continuous=True")
        self.net = BetaPolicy(obs_shape).to(device)
        self.optim = torch.optim.Adam(self.net.parameters(), lr=self.cfg["lr"], eps=1e-5)
        self.buffer = []

    @torch.no_grad()
    def act(self, obs, deterministic=False):
        x = torch.as_tensor(obs, device=self.device)[None]
        if deterministic:
            return self.net(x)[0].cpu().numpy()
        alpha, beta, value = self.net.parameters_at(x)
        dist = Beta(alpha, beta)
        sample = dist.sample()
        self.pending = (sample[0].cpu().numpy(), dist.log_prob(sample).sum().item(), value.item())
        return controls(sample)[0].cpu().numpy()

    def step(self, obs, action, reward, next_obs, terminated, truncated):
        self.buffer.append((obs, *self.pending, reward, next_obs, terminated, truncated))
        if len(self.buffer) < self.cfg["rollout"]:
            return {}
        metrics = self.update()
        self.buffer.clear()
        return metrics

    def update(self):
        c = self.cfg
        obs, samples, logs, values, rewards, successors, terms, truncs = zip(*self.buffer)
        def tensor(seq, dtype):
            return torch.as_tensor(np.stack(seq), device=self.device, dtype=dtype)
        x = tensor(obs, torch.uint8)
        u = tensor(samples, torch.float32)
        old_log = tensor(logs, torch.float32)
        with torch.no_grad():
            next_x = tensor(successors, torch.uint8)
            next_values = torch.cat([self.net.parameters_at(chunk)[2] for chunk in next_x.split(c["batch"])]).cpu().numpy()
        advantages = np.empty(len(rewards), dtype=np.float32)
        tail = 0.
        for i in reversed(range(len(rewards))):
            delta = rewards[i] * c["reward_scale"] - values[i]
            if not terms[i]:
                delta += c["gamma"] * next_values[i]
            carry = 0. if terms[i] or truncs[i] else tail
            tail = delta + c["gamma"] * c["gae_lambda"] * carry
            advantages[i] = tail
        adv = tensor(advantages, torch.float32)
        targets = adv + tensor(values, torch.float32)
        adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)
        losses = []
        for epoch in range(c["epochs"]):
            divergences = []
            for idx in torch.randperm(len(x), device=self.device).split(c["batch"]):
                a, b, pred = self.net.parameters_at(x[idx])
                dist = Beta(a, b)
                log_ratio = dist.log_prob(u[idx]).sum(-1) - old_log[idx]
                ratio = log_ratio.exp()
                clipped = ratio.clamp(1-c["clip"], 1+c["clip"])
                policy_loss = -torch.minimum(ratio * adv[idx], clipped * adv[idx]).mean()
                loss = policy_loss + 0.5 * F.mse_loss(pred, targets[idx])
                loss -= c["entropy_coef"] * dist.entropy().sum(-1).mean()
                self.optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5, error_if_nonfinite=True)
                self.optim.step()
                losses.append(loss.item())
                divergences.append((ratio.detach() - 1 - log_ratio.detach()).mean().item())
            if np.mean(divergences) > c["target_kl"]:
                break
        return dict(loss=float(np.mean(losses)), kl=float(np.mean(divergences)),
                    gas=float((2*u[:,1]-1).relu().mean()), brake=float((1-2*u[:,1]).relu().mean()))

    def export_policy(self):
        return self.net

    def state_dict(self):
        return dict(net=self.net.state_dict(), optim=self.optim.state_dict())

    def load_state_dict(self, state):
        self.net.load_state_dict(state["net"])
        self.optim.load_state_dict(state["optim"])
        self.buffer.clear()  # A resumed environment starts a new episode.

In [ ]:
# 실제 학습 전, 합성 관측으로 PPO 업데이트와 제출 정책 출력을 점검합니다.
import torch, gymnasium as gym, numpy as np
from agents.beta_ppo import BetaPPO
space = gym.spaces.Box(np.array([-1,0,0], np.float32), np.ones(3, np.float32))
test = BetaPPO((4,96,96), space, dict(rollout=4, batch=2, epochs=1), 'cpu')
obs = np.zeros((4,96,96), dtype=np.uint8)
for i in range(4):
    action = test.act(obs)
    assert space.contains(action) and action[1] * action[2] == 0
    stats = test.step(obs, action, 1., obs, i==1, i==3)
assert np.isfinite(stats['loss'])
np.testing.assert_allclose(test.act(obs, True), test.export_policy()(torch.from_numpy(obs)[None]).detach().numpy()[0])
print('합성 입력 검사 통과', stats)

In [ ]:
cmd = [sys.executable, '-u', 'train.py', '--agent', 'beta_ppo', '--seed', '42',
       '--budget', '300000', '--eval-every', '25000', '--run-dir', str(run)]
if (run/'latest.pt').exists():
    cmd.append('--resume')
elif run.exists() and any(run.iterdir()):
    raise RuntimeError('체크포인트가 없는 기존 run입니다. 새 전용 경로를 지정하세요.')
subprocess.run(cmd, check=True)

In [ ]:
# 팀 이름 입력 → 행동 일치율 검사 → 공식 공개 트랙 평가 → 최종 ZIP
import re, json
from evaluate import load_agent
from export import export, verify
team = input('등록한 팀 이름: ').strip()
if not team or re.search(r'[\\/:*?"<>|]', team) or team in ('.', '..'):
    raise ValueError('팀 이름을 확인하세요.')
onnx_path = run / (team + '_beta.onnx')
agent = load_agent(str(run/'best.pt'), torch.device('cpu'))
export(agent, str(onnx_path))
rate, steps = verify(agent, str(onnx_path))
print(f'행동 일치율 {rate:.1%} ({steps} steps)')
if rate < .95:
    raise RuntimeError('행동 일치율 기준 미달')
subprocess.run([sys.executable, 'check_submission.py', '--onnx', str(onnx_path),
                '--seeds', '1000-1019', '--json', str(run/'public_eval.json')], check=True)
config = json.loads((run/'config.json').read_text())
assert config['budget'] == 300000 and config['agent'] == 'beta_ppo' and config['seed'] == 42
submission = run / (team + '_submission.zip')
with zipfile.ZipFile(submission, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in [*manifest, 'agents/beta_ppo.py']:
        archive.write(root/name, name)
    for name in ['config.json','train.csv','eval.csv','best.pt','public_eval.json']:
        archive.write(run/name, 'runs/beta_ppo_s42/'+name)
    archive.write(onnx_path, onnx_path.name)
    archive.writestr('REPRODUCE.txt', 'pip install -r requirements.txt\npython train.py --agent beta_ppo --seed 42 --budget 300000 --eval-every 25000 --run-dir runs/beta_ppo_s42\n')
print(submission)
files.download(str(submission))